<a href="https://colab.research.google.com/github/shrivash-raha/gemma-3-1b-lora-tuning/blob/main/gemma-3-1b-fine-tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes torchao>=0.16.0

In [78]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel
)

from datasets import load_dataset

In [79]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [21]:
total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Total Parameters: {total_params:,}")

Total Parameters: 494,032,768


In [28]:
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable Parameters: {trainable_params:,}")

Trainable Parameters: 494,032,768


In [29]:
for name, module in model.named_modules():
    if "proj" in name:
        print(name)

model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.self_attn.o_proj
model.layers.2.mlp.gate_proj
model.layers.2.mlp.up_proj
model.layers.2.mlp.down_proj
model.layers.3.self_attn.q_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.3.self_attn.o_proj
model.layers.3.mlp.gate_proj
model.layers.3.mlp.up_proj
model.layers.3.mlp.down_proj
model.layers.4.self_attn.q_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.4.self_attn.o_proj
model.layers.4.mlp.g

In [42]:
for name, module in model.named_modules():
    if "proj" in name:
        print(f"Module: {name}")
        for param_name, param in module.named_parameters():
            print(f"  Parameter: {param_name}, Shape: {param.shape}")

Module: model.layers.0.self_attn.q_proj
  Parameter: weight, Shape: torch.Size([896, 896])
  Parameter: bias, Shape: torch.Size([896])
Module: model.layers.0.self_attn.k_proj
  Parameter: weight, Shape: torch.Size([128, 896])
  Parameter: bias, Shape: torch.Size([128])
Module: model.layers.0.self_attn.v_proj
  Parameter: weight, Shape: torch.Size([128, 896])
  Parameter: bias, Shape: torch.Size([128])
Module: model.layers.0.self_attn.o_proj
  Parameter: weight, Shape: torch.Size([896, 896])
Module: model.layers.0.mlp.gate_proj
  Parameter: weight, Shape: torch.Size([4864, 896])
Module: model.layers.0.mlp.up_proj
  Parameter: weight, Shape: torch.Size([4864, 896])
Module: model.layers.0.mlp.down_proj
  Parameter: weight, Shape: torch.Size([896, 4864])
Module: model.layers.1.self_attn.q_proj
  Parameter: weight, Shape: torch.Size([896, 896])
  Parameter: bias, Shape: torch.Size([896])
Module: model.layers.1.self_attn.k_proj
  Parameter: weight, Shape: torch.Size([128, 896])
  Parameter: 

In [71]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [80]:
model = get_peft_model(
    model,
    lora_config
)

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported